
# Week 5: Semantic Search with Embeddings

## Objective

The goal of Week 5 is to build a semantic search system
for real estate listings using sentence embeddings and FAISS.

Traditional keyword search looks for exact or similar words.
Semantic search focuses on the meaning of a query, even when
the property description uses different wording.

For this project, I will:
1. Load real estate listing remarks from MySQL.
2. Generate 384-dimensional sentence embeddings.
3. Build a FAISS index for similarity search.
4. Compare semantic search with BM25 keyword search.
5. Measure search latency.
6. Evaluate the relevance of search results.

The main goal is to determine whether semantic search provides
more relevant results than traditional keyword matching.


In [1]:

# Import libraries used throughout the notebook.
# pandas helps us organize and display the results.
# Path helps us locate our project files.

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time

# Find the main project folder.
# Our notebook is inside notebooks/, so we may need
# to move one folder up to import scripts/.

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP



## Loading the Dataset

I am using the real_estate database in MySQL, specifically
the rets_property table.

The main feature for semantic search is L_Remarks, which
contains written descriptions of properties.

I will initially use 10,000 listings to build and evaluate
the search system before expanding to the full dataset.



In [2]:

import mysql.connector
import getpass

# Connect to our existing real estate database.
# getpass hides the password while we type it.

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=getpass.getpass("MySQL password: "),
    database="real_estate"
)

# Select property descriptions and useful information.
# We remove listings with missing or empty remarks.

query = """
SELECT
    L_City,
    L_SystemPrice,
    L_Remarks
FROM rets_property
WHERE L_Remarks IS NOT NULL
  AND TRIM(L_Remarks) <> ''
LIMIT 10000
"""

cursor = conn.cursor()

try:
    cursor.execute(query)

    rows = cursor.fetchall()
    columns = cursor.column_names

    df = pd.DataFrame(rows, columns=columns)

finally:
    cursor.close()
    conn.close()

# Reset the row numbers so they match the FAISS positions.
df = df.reset_index(drop=True)

print("Total listings:", len(df))
display(df.head())


Total listings: 10000


,L_City,L_SystemPrice,L_Remarks
0,Beverly Hills,3950000,Opportunity to reimagine a Classic 70's Archit...
1,Lathrop,775000,Step into comfort and functionality with this ...
2,Sherman Oaks,750000,WOW! NOW REDUCED AND THE SELLER MAY FINANCE! C...
3,Mountain House,1122426,Plan 3 features an open floor plan with covere...
4,Mountain House,1128292,Plan 2 features an open floor plan with covere...



## Sentence Embeddings and FAISS

I will use the all-MiniLM-L6-v2 model to convert each
property description into a 384-dimensional vector.

The model represents text numerically so descriptions
with similar meanings can be compared.

FAISS stores these vectors and helps retrieve the most
similar listings efficiently.


In [3]:

from scripts.semantic_search import SemanticSearcher

# Create the semantic search model.
searcher = SemanticSearcher()

# Get property descriptions as a Python list.
remarks = df["L_Remarks"].fillna("").tolist()

# Generate embeddings and add them to FAISS.
# This may take several minutes for 10,000 listings.
searcher.build_index(remarks)

print("Indexed listings:", searcher.index.ntotal)
print("Embedding dimensions:", searcher.index.d)


W1025 18:21:54.664000 11860 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luoxu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (8 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Indexed listings: 10000
Embedding dimensions: 384


In [6]:
# Search for the five most relevant property listings
query = "Spacious home with a swimming pool and backyard"
results = searcher.search(query, top_k=5)

# Display the first result to understand its structure
print("First result:", results[0])

# Check the data type of the result
print("Result type:", type(results[0]))

First result: {'row_index': 8491, 'listing': 'A beautiful, well maintained pool home with 3 beds, 2 baths and a large den area, plus two living areas and a dining room. This home is filled with light and has  original wood floors.  There is also a large kitchen with Corian counter tops, eat-in area and views of the back yard area.  AC system, newer windows and upgraded electrical.  There is a long driveway and the two car garage with ample storage.  Behind the garage is a large, usable outdoor area. The pool area is  private. The views from the home are stunning.   This home need a little love, but would be the perfect home for a young family and  is located in Hawthorne Elementary school district.  This is a GREAT opportunity.  Show now before it’s gone!', 'similarity': 0.7654404640197754}
Result type: <class 'dict'>


In [7]:
# Define the property features we want to search for
query = "Spacious home with a swimming pool and backyard"

# Retrieve the five most relevant listings using semantic search
results = searcher.search(query, top_k=5)

# Loop through each search result, starting the numbering at 1
for i, result in enumerate(results, start=1):

    # Extract the listing's position in the indexed dataset
    row_index = result["row_index"]

    # Extract the property description
    remark = result["listing"]

    # Extract the cosine similarity score
    score = result["similarity"]

    # Display the search result in a readable format
    print(f"\nResult {i}")
    print(f"Row Index: {row_index}")
    print(f"Similarity Score: {score:.4f}")
    print(f"Description: {remark[:300]}")


Result 1
Row Index: 8491
Similarity Score: 0.7654
Description: A beautiful, well maintained pool home with 3 beds, 2 baths and a large den area, plus two living areas and a dining room. This home is filled with light and has  original wood floors.  There is also a large kitchen with Corian counter tops, eat-in area and views of the back yard area.  AC system, n

Result 2
Row Index: 9047
Similarity Score: 0.7459
Description: Luxury home custom built with swimming pool

Result 3
Row Index: 1592
Similarity Score: 0.7293
Description: ***POOL HOME with Price Improvement!*** Beautiful and spacious Pool Home with a Price Improvement! 4-Bedroom, 2-Bath home that offers comfort, space, and versatility.  The tastefully converted garage provides a 4th bedroom with its own Full Restroom and Separate/Private side entrance, making it idea

Result 4
Row Index: 6481
Similarity Score: 0.7276
Description: Step into the appeal of this stunning two-story home, complete with a spacious two-car garage and 

In [9]:
# Retrieve the original property descriptions from the SemanticSearcher object
# These are the same 10,000 listings used to build the FAISS index
remarks_list = searcher.listings

# Verify that the descriptions were retrieved successfully
print(f"Total listings: {len(remarks_list)}")

# Display the first listing to confirm the data looks correct
print(f"First listing: {remarks_list[0][:200]}")

Total listings: 10000
First listing: Opportunity to reimagine a Classic 70's Architectural property located on a quiet cul-de-sac in BHPO. This home was build in 1973  in the Mid Century era. The breakfast and kitchen areas provide pictu


In [10]:
# Import the BM25 algorithm for keyword-based search
from rank_bm25 import BM25Okapi

# Import regular expressions for basic text preprocessing
import re

# Import NumPy to rank listings by their BM25 scores
import numpy as np


# Define a function to tokenize text into lowercase words
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


# Convert each property description into a list of words
tokenized_listings = [tokenize(text) for text in remarks_list]

# Build the BM25 search model using the same 10,000 listings
bm25 = BM25Okapi(tokenized_listings)


# Define a function to search listings using BM25
def bm25_search(query, top_k=5):

    # Convert the search query into lowercase words
    query_tokens = tokenize(query)

    # Calculate BM25 relevance scores for all listings
    scores = bm25.get_scores(query_tokens)

    # Identify the listings with the highest BM25 scores
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Store each matching listing and its BM25 score
    results = []

    for i in top_indices:
        results.append({
            "row_index": int(i),
            "listing": remarks_list[i],
            "score": float(scores[i])
        })

    # Return the ranked search results
    return results

In [11]:
# Define the same query used in semantic search
query = "Spacious home with a swimming pool and backyard"

# Retrieve the top five keyword-based search results
results = bm25_search(query, top_k=5)

# Display each result's score and property description
for i, result in enumerate(results, start=1):

    # Print the ranking position
    print(f"\nResult {i}")

    # Display the BM25 relevance score
    print(f"BM25 Score: {result['score']:.4f}")

    # Display the first 300 characters of the listing
    print(f"Description: {result['listing'][:300]}")


Result 1
BM25 Score: 22.1211
Description: Turnkey newer-construction corner-unit home nestled in a desirable community with 4 bedrooms and 3 bathrooms. This clean and inviting home features an open floor plan, granite countertops, stainless steel appliances, and hard-surface flooring throughout the main living areas. The backyard offers a s

Result 2
BM25 Score: 21.4914
Description: Beautifully Remodeled Perris Home with Endless Potential! Seller will help with some closing cost.

Welcome to this beautifully remodeled home located on a great street in Perris, featuring a spacious front and backyard, a large, covered porch, and all-new landscaping that creates fantastic curb

Result 3
BM25 Score: 21.4065
Description: Located in the prestigious 55+ gated community of Four Seasons, this beautifully maintained home offers 24/7 security and is surrounded by scenic hills and the Hemet Golf Course. With over 2,000 sq ft of living space, this spacious floor plan features 2 bedrooms, 2 bathroo

In [12]:
# Define the same query for both search methods
query = "Spacious home with a swimming pool and backyard"

# Retrieve the top five matches from each search method
semantic_results = searcher.search(query, top_k=5)
bm25_results = bm25_search(query, top_k=5)


# Display all semantic search results without truncating descriptions
print("=" * 60)
print("SEMANTIC SEARCH RESULTS")
print("=" * 60)

for i, result in enumerate(semantic_results, start=1):
    # Print the listing's rank, source index, and similarity score
    print(f"\nResult {i}")
    print(f"Row Index: {result['row_index']}")
    print(f"Similarity Score: {result['similarity']:.4f}")

    # Print the complete description for relevance evaluation
    print(f"Description: {result['listing']}")


# Display all BM25 results without truncating descriptions
print("\n" + "=" * 60)
print("BM25 SEARCH RESULTS")
print("=" * 60)

for i, result in enumerate(bm25_results, start=1):
    # Print the listing's rank, source index, and BM25 score
    print(f"\nResult {i}")
    print(f"Row Index: {result['row_index']}")
    print(f"BM25 Score: {result['score']:.4f}")

    # Print the complete description for relevance evaluation
    print(f"Description: {result['listing']}")
    

SEMANTIC SEARCH RESULTS

Result 1
Row Index: 8491
Similarity Score: 0.7654
Description: A beautiful, well maintained pool home with 3 beds, 2 baths and a large den area, plus two living areas and a dining room. This home is filled with light and has  original wood floors.  There is also a large kitchen with Corian counter tops, eat-in area and views of the back yard area.  AC system, newer windows and upgraded electrical.  There is a long driveway and the two car garage with ample storage.  Behind the garage is a large, usable outdoor area. The pool area is  private. The views from the home are stunning.   This home need a little love, but would be the perfect home for a young family and  is located in Hawthorne Elementary school district.  This is a GREAT opportunity.  Show now before it’s gone!

Result 2
Row Index: 9047
Similarity Score: 0.7459
Description: Luxury home custom built with swimming pool

Result 3
Row Index: 1592
Similarity Score: 0.7293
Description: ***POOL HOME with Pr

In [13]:
import numpy as np

# Manual relevance labels assigned to the five results
# 0 = Not relevant
# 1 = Partially relevant
# 2 = Highly relevant

semantic_labels = [2, 1, 2, 1, 0]
bm25_labels = [1, 1, 1, 1, 2]


# Calculate Precision@5
# A result is considered relevant if its score is at least 1
def precision_at_5(labels):
    return sum(score >= 1 for score in labels) / 5


# Calculate how many listings are highly relevant
def highly_relevant_count(labels):
    return sum(score == 2 for score in labels)


# Calculate average graded relevance on a 0–2 scale
def average_relevance(labels):
    return np.mean(labels)


# Display evaluation results for both methods
print("SEMANTIC SEARCH")
print(f"Precision@5: {precision_at_5(semantic_labels):.2f}")
print(f"Highly Relevant: {highly_relevant_count(semantic_labels)}")
print(f"Average Relevance: {average_relevance(semantic_labels):.2f}")

print("\nBM25 SEARCH")
print(f"Precision@5: {precision_at_5(bm25_labels):.2f}")
print(f"Highly Relevant: {highly_relevant_count(bm25_labels)}")
print(f"Average Relevance: {average_relevance(bm25_labels):.2f}")

SEMANTIC SEARCH
Precision@5: 0.80
Highly Relevant: 2
Average Relevance: 1.20

BM25 SEARCH
Precision@5: 1.00
Highly Relevant: 1
Average Relevance: 1.20


In [15]:
# Define five different real estate search queries
# Each query tests a different type of buyer preference

test_queries = [
    "Spacious home with a swimming pool and backyard",
    "Affordable starter home with updated kitchen",
    "Quiet family home near good schools",
    "Luxury waterfront property with scenic views",
    "Property with rental income potential"
]

# Confirm the number of test queries
print(f"Total test queries: {len(test_queries)}")

Total test queries: 5


In [16]:
import pandas as pd

# Create an empty list to store evaluation results
evaluation_data = []

# Loop through each of the five search queries
for query in test_queries:

    # Retrieve the top five semantic search results
    semantic_results = searcher.search(query, top_k=5)

    # Retrieve the top five BM25 keyword search results
    bm25_results = bm25_search(query, top_k=5)

    # Store the semantic search results
    for rank, result in enumerate(semantic_results, start=1):

        evaluation_data.append({
            "Query": query,
            "Method": "Semantic",
            "Rank": rank,
            "Row_Index": result["row_index"],
            "Description": result["listing"],
            "Search_Score": result["similarity"],
            "Relevance": None
        })

    # Store the BM25 keyword search results
    for rank, result in enumerate(bm25_results, start=1):

        evaluation_data.append({
            "Query": query,
            "Method": "BM25",
            "Rank": rank,
            "Row_Index": result["row_index"],
            "Description": result["listing"],
            "Search_Score": result["score"],
            "Relevance": None
        })

# Convert all results into a DataFrame
evaluation_df = pd.DataFrame(evaluation_data)

# Check that exactly 50 results were collected
print(f"Total evaluations: {len(evaluation_df)}")

# Display the first 10 results
display(evaluation_df.head(10))

Total evaluations: 50


,Query,Method,Rank,Row_Index,Description,Search_Score,Relevance
0,Spacious home with a swimming pool and backyard,Semantic,1,8491,"A beautiful, well maintained pool home with 3 ...",0.765440,None
1,Spacious home with a swimming pool and backyard,Semantic,2,9047,Luxury home custom built with swimming pool,0.745898,None
2,Spacious home with a swimming pool and backyard,Semantic,3,1592,***POOL HOME with Price Improvement!*** Beauti...,0.729347,None
3,Spacious home with a swimming pool and backyard,Semantic,4,6481,Step into the appeal of this stunning two-stor...,0.727564,None
4,Spacious home with a swimming pool and backyard,Semantic,5,2826,Beautiful Custom Home that sits on a great wat...,0.718168,None
5,Spacious home with a swimming pool and backyard,BM25,1,9691,Turnkey newer-construction corner-unit home ne...,22.121088,None
6,Spacious home with a swimming pool and backyard,BM25,2,4429,Beautifully Remodeled Perris Home with Endless...,21.491433,None
7,Spacious home with a swimming pool and backyard,BM25,3,7119,Located in the prestigious 55+ gated community...,21.406451,None
8,Spacious home with a swimming pool and backyard,BM25,4,756,"From gourmet cooking to golfing, living where ...",21.384571,None
9,Spacious home with a swimming pool and backyard,BM25,5,9237,"Stylishly updated 3-bedroom, 2-bath single-fam...",21.362385,None


In [18]:
from pathlib import Path

# Create the data directory if it does not already exist
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

# Define the output file location
output_file = output_dir / "week5_relevance_evaluation.xlsx"

# Export the 50 search results to Excel
evaluation_df.to_excel(output_file, index=False)

# Confirm the file was saved
print(f"Evaluation file saved to: {output_file.resolve()}")

Evaluation file saved to: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\notebooks\data\week5_relevance_evaluation.xlsx


In [19]:
# Import libraries for measuring execution time
import time
import numpy as np

# Define five different real estate search queries
test_queries = [
    "Spacious home with a swimming pool and backyard",
    "Affordable starter home with updated kitchen",
    "Quiet family home near good schools",
    "Luxury waterfront property with scenic views",
    "Property with rental income potential"
]

# Create an empty list to store search times in milliseconds
latencies = []

# Warm up the model before measuring performance
# This avoids including one-time initialization overhead
for query in test_queries:
    searcher.search(query, top_k=5)

# Repeat the benchmark 10 times for more reliable results
for _ in range(10):

    # Test each search query
    for query in test_queries:

        # Record the time immediately before searching
        start_time = time.perf_counter()

        # Perform semantic search across the indexed listings
        results = searcher.search(query, top_k=5)

        # Record the time immediately after searching
        end_time = time.perf_counter()

        # Convert seconds into milliseconds
        latency_ms = (end_time - start_time) * 1000

        # Store the measured latency
        latencies.append(latency_ms)

# Calculate performance statistics across all 50 searches
avg_latency = np.mean(latencies)
median_latency = np.median(latencies)
p95_latency = np.percentile(latencies, 95)
max_latency = np.max(latencies)

# Display benchmark results
print(f"Indexed listings: {searcher.index.ntotal}")
print(f"Total searches: {len(latencies)}")

print(f"\nAverage latency: {avg_latency:.2f} ms")
print(f"Median latency: {median_latency:.2f} ms")
print(f"95th percentile latency: {p95_latency:.2f} ms")
print(f"Maximum latency: {max_latency:.2f} ms")

# Check whether the average and 95th percentile meet the target
if avg_latency < 100 and p95_latency < 100:
    print("\nPASS: Average and P95 latency are below 100 ms.")
else:
    print("\nNOT MET: Average or P95 latency is 100 ms or higher.")

Indexed listings: 10000
Total searches: 50

Average latency: 39.18 ms
Median latency: 39.15 ms
95th percentile latency: 48.13 ms
Maximum latency: 53.01 ms

PASS: Average and P95 latency are below 100 ms.


In [20]:
import pandas as pd
import numpy as np

# Load the Excel file containing your manual relevance scores
evaluation_df = pd.read_excel(
    "data/week5_relevance_evaluation.xlsx"
)

# Check that all 50 results have been evaluated
print(f"Total evaluations: {len(evaluation_df)}")
print(f"Missing relevance scores: {evaluation_df['Relevance'].isna().sum()}")

# Display the first five rows to verify the data
display(evaluation_df.head())

# Confirm that all 50 rows have valid relevance scores
assert len(evaluation_df) == 50, "Expected 50 evaluations"
assert evaluation_df["Relevance"].isin([0, 1, 2]).all(), \
    "All relevance scores must be 0, 1, or 2"

Total evaluations: 50
Missing relevance scores: 0


,Query,Method,Rank,Row_Index,Description,Search_Score,Relevance
0,Spacious home with a swimming pool and backyard,Semantic,1,8491,"A beautiful, well maintained pool home with 3 ...",0.765440,2
1,Spacious home with a swimming pool and backyard,Semantic,2,9047,Luxury home custom built with swimming pool,0.745898,1
2,Spacious home with a swimming pool and backyard,Semantic,3,1592,***POOL HOME with Price Improvement!*** Beauti...,0.729347,2
3,Spacious home with a swimming pool and backyard,Semantic,4,6481,Step into the appeal of this stunning two-stor...,0.727564,2
4,Spacious home with a swimming pool and backyard,Semantic,5,2826,Beautiful Custom Home that sits on a great wat...,0.718168,0


In [21]:
# Calculate Precision@5
# A result is relevant if its manual score is 1 or 2
def precision_at_5(labels):
    return sum(score >= 1 for score in labels) / len(labels)


# Calculate Discounted Cumulative Gain
# Higher relevance scores and higher positions receive more weight
def dcg_at_5(labels):
    labels = np.array(labels)

    # Convert relevance labels into graded gains
    gains = (2 ** labels) - 1

    # Discount the relevance of lower-ranked results
    discounts = np.log2(np.arange(2, len(labels) + 2))

    return np.sum(gains / discounts)


# Calculate NDCG@5
# Compare the actual ranking against the ideal ordering
# of the five evaluated results
def ndcg_at_5(labels):
    actual_dcg = dcg_at_5(labels)

    # Sort relevance labels from highest to lowest
    ideal_labels = sorted(labels, reverse=True)
    ideal_dcg = dcg_at_5(ideal_labels)

    # Avoid division by zero if all results are irrelevant
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0


# Create an empty list to store each query's metrics
metrics = []

# Group results by search query and retrieval method
for (query, method), group in evaluation_df.groupby(
    ["Query", "Method"], sort=False
):

    # Preserve the original search-result ranking
    group = group.sort_values("Rank")

    # Retrieve manually assigned relevance scores
    labels = group["Relevance"].astype(int).tolist()

    # Calculate evaluation metrics
    metrics.append({
        "Query": query,
        "Method": method,
        "Precision@5": precision_at_5(labels),
        "Average_Relevance": np.mean(labels),
        "Highly_Relevant": sum(score == 2 for score in labels),
        "NDCG@5": ndcg_at_5(labels)
    })

# Convert metrics into a DataFrame
metrics_df = pd.DataFrame(metrics)

# Display performance for each query and method
display(metrics_df)

,Query,Method,Precision@5,Average_Relevance,Highly_Relevant,NDCG@5
0,Spacious home with a swimming pool and backyard,Semantic,0.8,1.4,3,0.941305
1,Spacious home with a swimming pool and backyard,BM25,0.8,1.2,2,0.678735
2,Affordable starter home with updated kitchen,Semantic,1.0,1.2,1,1.000000
3,Affordable starter home with updated kitchen,BM25,1.0,1.6,3,0.810696
4,Quiet family home near good schools,Semantic,1.0,1.6,3,0.980771
5,Quiet family home near good schools,BM25,0.4,0.4,0,0.624051
6,Luxury waterfront property with scenic views,Semantic,1.0,1.4,2,0.935510
7,Luxury waterfront property with scenic views,BM25,0.8,1.2,2,0.931225
8,Property with rental income potential,Semantic,1.0,1.6,3,0.968615
9,Property with rental income potential,BM25,0.8,1.6,4,0.982892


In [22]:
# Calculate average performance across all five queries
summary_df = metrics_df.groupby("Method")[
    ["Precision@5", "Average_Relevance", "Highly_Relevant", "NDCG@5"]
].mean().round(3)

# Display the overall evaluation results
print("SEMANTIC SEARCH VS. BM25")
display(summary_df)

# Save results for your Week 5 report
summary_df.to_csv(
    "data/week5_evaluation_summary.csv"
)

SEMANTIC SEARCH VS. BM25


,Precision@5,Average_Relevance,Highly_Relevant,NDCG@5
Method,,,,
BM25,0.76,1.20,2.2,0.806
Semantic,0.96,1.44,2.4,0.965
